In [1]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

#provider = TracerProvider()
#provider.add_span_processor(
#    SimpleSpanProcessor(ConsoleSpanExporter())
#)
#trace.set_tracer_provider(provider)

#tracer = trace.get_tracer("llm-zoomcamp")

In [4]:
from starter import index, client

Q1 and Q2

###### Made some changes to not repeat and confuse myself with the functions

In [5]:
from rag_helper import RAGBase

class RAGTraced(RAGBase):

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            usage = response.usage

            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)

            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

In [9]:
rag=RAGTraced(index=index,llm_client=client)

In [10]:
que="How does the agentic loop keep calling the model until it stops?"

In [11]:
ans=rag.rag(que)
print(ans)

The loop keeps calling the model inside a `while True` block. After each response, it checks whether the model returned any `function_call` items:

- if there is a function call, the code runs the tool, appends the tool output to `messages`, and loops again
- if there are no function calls, it breaks out of the loop

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In other words, it keeps going until the model returns a final message with no more tool calls.


Q3

In [12]:
#data from above, copied
llm_span={
    "start_time": "2026-07-18T18:49:50.446263Z",
    "end_time": "2026-07-18T18:49:51.996968Z",
}

from datetime import datetime

start = datetime.fromisoformat(llm_span["start_time"].replace("Z", "+00:00"))
end = datetime.fromisoformat(llm_span["end_time"].replace("Z", "+00:00"))

duration = end - start

print(duration)
print(duration.total_seconds())
print(duration.total_seconds() * 1000)

0:00:01.550705
1.550705
1550.705


Q4

In [7]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

#### Restart the kernel

In [8]:
provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [13]:
ans=rag.rag(que)
print(ans)

The loop keeps calling the model by checking whether the model returned any `function_call`s.

- It sends the current `messages` to the model.
- If the response includes a function call, the code runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
- If there are no function calls in that turn, the loop breaks.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In short: it repeats until the model returns a final message with no more tool calls.


Q5

In [14]:
conn = sqlite3.connect("traces.db")
conn.execute("SELECT name FROM spans").fetchall()

[('search',), ('llm',), ('rag',), ('search',), ('llm',), ('rag',)]

In [19]:
conn.execute(
    """
    SELECT name, SUM(end_time-start_time) AS duration 
    FROM spans 
    WHERE name!='rag'
    GROUP BY name 
    """
    ).fetchall()

[('llm', 4718429640), ('search', 5773311)]

Q6

In [20]:
for _ in range(2): #Already have two executions!
    rag.rag(que)

In [21]:
conn.execute("SELECT * FROM spans").fetchall()

[('search', 1784401874370134692, 1784401874372736424, None, None, None),
 ('llm', 1784401874380235378, 1784401877733487199, 7111, 118, None),
 ('rag', 1784401874370024212, 1784401877738939505, None, None, None),
 ('search', 1784401893560726476, 1784401893563898055, None, None, None),
 ('llm', 1784401893568830948, 1784401894934008767, 7111, 121, None),
 ('rag', 1784401893560665905, 1784401894936199708, None, None, None),
 ('search', 1784402589921464234, 1784402589925659647, None, None, None),
 ('llm', 1784402589929967080, 1784402591512189513, 7111, 133, None),
 ('rag', 1784402589921422821, 1784402591515469296, None, None, None),
 ('search', 1784402591518039457, 1784402591519602472, None, None, None),
 ('llm', 1784402591522340449, 1784402593433577612, 7111, 117, None),
 ('rag', 1784402591517993160, 1784402593438176240, None, None, None)]

In [22]:
import pandas as pd

df=pd.read_sql_query("SELECT * FROM spans", conn)
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784401874370134692,1784401874372736424,NaN,NaN,None
1,llm,1784401874380235378,1784401877733487199,7111.0,118.0,None
2,rag,1784401874370024212,1784401877738939505,NaN,NaN,None
3,search,1784401893560726476,1784401893563898055,NaN,NaN,None
4,llm,1784401893568830948,1784401894934008767,7111.0,121.0,None
5,rag,1784401893560665905,1784401894936199708,NaN,NaN,None
6,search,1784402589921464234,1784402589925659647,NaN,NaN,None
7,llm,1784402589929967080,1784402591512189513,7111.0,133.0,None
8,rag,1784402589921422821,1784402591515469296,NaN,NaN,None
9,search,1784402591518039457,1784402591519602472,NaN,NaN,None


In [24]:
df[df["name"]=='llm']['input_tokens']

1     7111.0
4     7111.0
7     7111.0
10    7111.0
Name: input_tokens, dtype: float64